In [1]:
from pathlib import Path
import tensorflow as tf
import tensorflow_io as tfio
import pandas as pd


2026-04-17 21:03:59.192261: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-17 21:03:59.321722: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-04-17 21:03:59.321750: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-04-17 21:03:59.341718: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-04-17 21:03:59.385305: I tensorflow/core/platform/cpu_feature_guar

In [2]:
# Se configuran las rutas y se crean las carpetas para guardar la nueva distribución de datos.
# Dataset fuente actual:"
#                       Training/"clase"/*.wav
#                       Validation/"clase"/*.wa
SOURCE_ROOT = Path.cwd()
TRAIN_SRC = SOURCE_ROOT / "Training"
VAL_SRC = SOURCE_ROOT / "Validation"

DERIVED_ROOT = SOURCE_ROOT / "dataset_yamnet"
AUDIO_ROOT = DERIVED_ROOT / "audio"
META_ROOT = DERIVED_ROOT / "metadata"

# Sample rate acorde a lo que ingiere YAMNet
TARGET_SR = 16000
SEED = 42

for split in ["train", "validation", "test"]:
    (AUDIO_ROOT / split).mkdir(parents=True, exist_ok=True)

META_ROOT.mkdir(parents=True, exist_ok=True)

tf.random.set_seed(SEED)


In [29]:
# Se definen funciones para re-procesar los segmentos de audio.
# Lectura, resampleo a 16k y clipleado

def load_wav_16k_mono(file_path):
    """
    Lee WAV, fuerza mono y remuestrea a 16 kHz.
    """
    audio_binary = tf.io.read_file(file_path)
    waveform, sample_rate = tf.audio.decode_wav(audio_binary, desired_channels=1)
    waveform = tf.squeeze(waveform, axis=-1)
    sample_rate = tf.cast(sample_rate, tf.int64)
    waveform = tfio.audio.resample(waveform, rate_in=sample_rate, rate_out=TARGET_SR)
    waveform = tf.clip_by_value(waveform, -1.0, 1.0)
    return waveform


def save_wav_16k_mono(waveform, output_path):
    """
    Guarda un tensor 1D float32 como WAV mono 16 kHz.
    """
    waveform = tf.expand_dims(waveform, axis=-1)  # [samples, 1]
    wav_bytes = tf.audio.encode_wav(waveform, sample_rate=TARGET_SR)
    tf.io.write_file(str(output_path), wav_bytes)


In [3]:
# Se definen funciones pare el indexado del dataset

def list_class_dirs(split_dir):
    """
    Lista carpetas de clase dentro de un split original.
    """
    class_names = tf.io.gfile.listdir(str(split_dir))
    class_names = sorted([
        c for c in class_names
        if tf.io.gfile.isdir(str(split_dir / c))
    ])
    return class_names


def collect_split_files(split_dir, source_split):
    """
    Recorre un split original y arma una tabla de archivos.
    """
    rows = []

    class_names = list_class_dirs(split_dir)

    for class_name in class_names:
        pattern = str(split_dir / class_name / "*.wav")
        files = tf.io.gfile.glob(pattern)

        for f in files:
            rows.append({
                "source_path": f,
                "source_split": source_split,
                "class_name": class_name,
            })

    return pd.DataFrame(rows)



In [10]:
# Se construyen un dataframe con el path de los archivos y su clase.

df_train_src = collect_split_files(TRAIN_SRC, "Training")
df_val_src = collect_split_files(VAL_SRC, "Validation")

df_source = pd.concat([df_train_src, df_val_src], ignore_index=True)

print("Archivos fuente:", len(df_source))
print(df_source["source_split"].value_counts())
print("Cantidad de clases:", df_source["class_name"].nunique())
df_source.head

Archivos fuente: 6871
source_split
Training      4805
Validation    2066
Name: count, dtype: int64
Cantidad de clases: 14


<bound method NDFrame.head of                                             source_path source_split  \
0     /home/jorge/Documents/CEIA/tp_vc2/Training/Fri...     Training   
1     /home/jorge/Documents/CEIA/tp_vc2/Training/Fri...     Training   
2     /home/jorge/Documents/CEIA/tp_vc2/Training/Fri...     Training   
3     /home/jorge/Documents/CEIA/tp_vc2/Training/Fri...     Training   
4     /home/jorge/Documents/CEIA/tp_vc2/Training/Fri...     Training   
...                                                 ...          ...   
6866  /home/jorge/Documents/CEIA/tp_vc2/Validation/T...   Validation   
6867  /home/jorge/Documents/CEIA/tp_vc2/Validation/T...   Validation   
6868  /home/jorge/Documents/CEIA/tp_vc2/Validation/T...   Validation   
6869  /home/jorge/Documents/CEIA/tp_vc2/Validation/T...   Validation   
6870  /home/jorge/Documents/CEIA/tp_vc2/Validation/T...   Validation   

                        class_name  
0     Fringillidae_Serinus_serinus  
1     Fringillidae_Serinus_seri

In [23]:
# Se define función para separar el set de validación en test y validación 50/50

def split_validation_half_per_class(df_validation, seed=42):
    """
    Divide el Validation original en validation y test, clase por clase.
    """
    val_parts = []
    test_parts = []

    for class_name, group in df_validation.groupby("class_name", sort=True):
        group = group.sample(frac=1, random_state=seed).reset_index(drop=True)

        n_total = len(group)
        n_val = n_total // 2

        val_group = group.iloc[:n_val].copy()
        test_group = group.iloc[n_val:].copy()

      
        val_parts.append(val_group)
        test_parts.append(test_group)

    df_validation_new = pd.concat(val_parts, ignore_index=True) if val_parts else pd.DataFrame()
    df_test_new = pd.concat(test_parts, ignore_index=True) if test_parts else pd.DataFrame()

    return df_validation_new, df_test_new

In [27]:
df_validation_old = df_source[df_source["source_split"] == "Validation"].copy()
df_validation_new, df_test_new = split_validation_half_per_class(df_validation_old, seed=SEED)

df_train_final = df_source[df_source["source_split"] == "Training"].copy()
df_train_final["split"] = "train"

df_validation_new["split"] = "validation"
df_test_new["split"] = "test"

df_final = pd.concat(
    [df_train_final, df_validation_new, df_test_new],
    ignore_index=True
)

print("\nDistribución final por split:")
print(df_final["split"].value_counts())
df_final


Distribución final por split:
split
train         4805
test          1037
validation    1029
Name: count, dtype: int64


,source_path,source_split,class_name,split
0,/home/jorge/Documents/CEIA/tp_vc2/Training/Fri...,Training,Fringillidae_Serinus_serinus,train
1,/home/jorge/Documents/CEIA/tp_vc2/Training/Fri...,Training,Fringillidae_Serinus_serinus,train
2,/home/jorge/Documents/CEIA/tp_vc2/Training/Fri...,Training,Fringillidae_Serinus_serinus,train
3,/home/jorge/Documents/CEIA/tp_vc2/Training/Fri...,Training,Fringillidae_Serinus_serinus,train
4,/home/jorge/Documents/CEIA/tp_vc2/Training/Fri...,Training,Fringillidae_Serinus_serinus,train
...,...,...,...,...
6866,/home/jorge/Documents/CEIA/tp_vc2/Validation/T...,Validation,Turdidae_Catharus_ustulatus,test
6867,/home/jorge/Documents/CEIA/tp_vc2/Validation/T...,Validation,Turdidae_Catharus_ustulatus,test
6868,/home/jorge/Documents/CEIA/tp_vc2/Validation/T...,Validation,Turdidae_Catharus_ustulatus,test
6869,/home/jorge/Documents/CEIA/tp_vc2/Validation/T...,Validation,Turdidae_Catharus_ustulatus,test


In [28]:
df_final[df_final["split"]=="test"]

,source_path,source_split,class_name,split
5834,/home/jorge/Documents/CEIA/tp_vc2/Validation/F...,Validation,Fringillidae_Serinus_serinus,test
5835,/home/jorge/Documents/CEIA/tp_vc2/Validation/F...,Validation,Fringillidae_Serinus_serinus,test
5836,/home/jorge/Documents/CEIA/tp_vc2/Validation/F...,Validation,Fringillidae_Serinus_serinus,test
5837,/home/jorge/Documents/CEIA/tp_vc2/Validation/F...,Validation,Fringillidae_Serinus_serinus,test
5838,/home/jorge/Documents/CEIA/tp_vc2/Validation/F...,Validation,Fringillidae_Serinus_serinus,test
...,...,...,...,...
6866,/home/jorge/Documents/CEIA/tp_vc2/Validation/T...,Validation,Turdidae_Catharus_ustulatus,test
6867,/home/jorge/Documents/CEIA/tp_vc2/Validation/T...,Validation,Turdidae_Catharus_ustulatus,test
6868,/home/jorge/Documents/CEIA/tp_vc2/Validation/T...,Validation,Turdidae_Catharus_ustulatus,test
6869,/home/jorge/Documents/CEIA/tp_vc2/Validation/T...,Validation,Turdidae_Catharus_ustulatus,test


In [30]:
# Se mapean las clases
# 
class_names = sorted(df_final["class_name"].unique().tolist())
class_to_id = {name: idx for idx, name in enumerate(class_names)}

df_class_mapping = pd.DataFrame({
    "class_id": list(class_to_id.values()),
    "class_name": list(class_to_id.keys())
}).sort_values("class_id")

df_final["class_id"] = df_final["class_name"].map(class_to_id)

In [31]:
df_final

,source_path,source_split,class_name,split,class_id
0,/home/jorge/Documents/CEIA/tp_vc2/Training/Fri...,Training,Fringillidae_Serinus_serinus,train,0
1,/home/jorge/Documents/CEIA/tp_vc2/Training/Fri...,Training,Fringillidae_Serinus_serinus,train,0
2,/home/jorge/Documents/CEIA/tp_vc2/Training/Fri...,Training,Fringillidae_Serinus_serinus,train,0
3,/home/jorge/Documents/CEIA/tp_vc2/Training/Fri...,Training,Fringillidae_Serinus_serinus,train,0
4,/home/jorge/Documents/CEIA/tp_vc2/Training/Fri...,Training,Fringillidae_Serinus_serinus,train,0
...,...,...,...,...,...
6866,/home/jorge/Documents/CEIA/tp_vc2/Validation/T...,Validation,Turdidae_Catharus_ustulatus,test,13
6867,/home/jorge/Documents/CEIA/tp_vc2/Validation/T...,Validation,Turdidae_Catharus_ustulatus,test,13
6868,/home/jorge/Documents/CEIA/tp_vc2/Validation/T...,Validation,Turdidae_Catharus_ustulatus,test,13
6869,/home/jorge/Documents/CEIA/tp_vc2/Validation/T...,Validation,Turdidae_Catharus_ustulatus,test,13


In [32]:
export_rows = []

for i, row in df_final.reset_index(drop=True).iterrows():
    source_path = row["source_path"]
    split = row["split"]
    class_name = row["class_name"]
    class_id = row["class_id"]

    segment_id = f"seg_{i:06d}"
    out_name = f"{segment_id}.wav"
    out_path = AUDIO_ROOT / split / out_name

    # Preparación del audio para YAMNet
    waveform = load_wav_16k_mono(source_path)
    save_wav_16k_mono(waveform, out_path)

    export_rows.append({
        "segment_id": segment_id,
        "filepath": str(out_path),
        "split": split,
        "class_id": class_id,
        "label": class_name,
        "source_path": source_path,
        "source_split": row["source_split"],
    })

df_exported = pd.DataFrame(export_rows)

print("\nArchivos exportados:", len(df_exported))
print(df_exported["split"].value_counts())




2026-04-18 03:09:23.700178: I tensorflow_io/core/kernels/cpu_check.cc:128] Your CPU supports instructions that this TensorFlow IO binary was not compiled to use: SSE3 SSE4.1 SSE4.2 AVX AVX2 AVX512F FMA
2026-04-18 03:09:24.695577: I external/local_tsl/tsl/platform/default/subprocess.cc:304] Start cannot spawn child process: No such file or directory
2026-04-18 03:09:24.700439: W external/local_xla/xla/stream_executor/gpu/asm_compiler.cc:225] Falling back to the CUDA driver for PTX compilation; ptxas does not support CC 12.0
2026-04-18 03:09:24.700451: W external/local_xla/xla/stream_executor/gpu/asm_compiler.cc:228] Used ptxas at /home/jorge/.venvs/vc2/lib/python3.10/site-packages/tensorflow/python/platform/../../../nvidia/cuda_nvcc/bin/ptxas
2026-04-18 03:09:24.700491: W tensorflow/compiler/mlir/tools/kernel_gen/transforms/gpu_kernel_to_blob_pass.cc:191] Failed to compile generated PTX with ptxas. Falling back to compilation by driver.
2026-04-18 03:09:24.700498: W tensorflow/compiler/


Archivos exportados: 6871
split
train         4805
test          1037
validation    1029
Name: count, dtype: int64


In [34]:
# Se exporta información en tablas 
df_exported.to_csv(META_ROOT / "all_segments.csv", index=False)
df_exported[df_exported["split"] == "train"].to_csv(META_ROOT / "train.csv", index=False)
df_exported[df_exported["split"] == "validation"].to_csv(META_ROOT / "validation.csv", index=False)
df_exported[df_exported["split"] == "test"].to_csv(META_ROOT / "test.csv", index=False)
df_class_mapping.to_csv(META_ROOT / "class_mapping.csv", index=False)

print("\nDataset derivado generado en:")
print(DERIVED_ROOT)


Dataset derivado generado en:
/home/jorge/Documents/CEIA/tp_vc2/dataset_yamnet
